# 2.7 SIMT 离散类矢量算子示例（Gather 算子）
本入门示例基于 Ascend C SIMT 实现一维 Gather 算子，带你快速上手实践，涵盖 Device 端核函数实现、Host 端调用以及编译运行的完整流程，帮助开发者建立整体认知。

**Gather 算子功能介绍**：数学表达式为 `output[i] = input[index[i]]`，计算逻辑为根据索引张量 index 中的每个元素，从一维输入向量 input 中采集对应位置的数据，并写入输出张量 output。

> SIMT 是 Ascend 950PR/Ascend 950DT 专属执行模式，编译时需加 `--enable-simt` 选项。


## 算子设计
**Device 端核函数**：
- 通过 `__global__` 修饰符声明核函数。
- 数据划分：使用内置变量 `threadIdx`、`blockIdx`、`blockDim` 计算线程索引，为每个线程分配需要处理的数据元素。
- 数据搬入/搬出：无需额外接口，直接通过指针访问即可。
- 数据计算：根据 index 中的索引值及操作符 `[]` 读取输入数据。

**Host 端运行时**：
- 使用 `aclrtMallocHost` 分配 Host Memory，`aclrtMalloc` 分配 Device Memory。
- 通过 `<<<grid_dim, block_dim, dynamic_memory_size, stream>>>` 语法糖启动核函数（4 个参数，比 SIMD 多一个 `block_dim`）。
- 调用 `aclrtSynchronizeStream` 等待任务完成，`aclrtMemcpy` 将结果拷回 Host Memory。


## Device 端 Kernel 实现
后缀名为 `*.asc` 的代码文件包含 Host 端与 Device 端代码。Device 端核函数实现如下：

```cpp
__global__ void gather_1d_custom(float* input, int32_t* index, float* output, uint64_t index_total_length)
{
    // Calculate global thread ID
    int32_t idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Maps to the index of output tensor
    if (idx >= index_total_length) {
        return;
    }
    output[idx] = input[index[idx]];
}
```

**关键说明**：
- `blockIdx.x`/`blockDim.x`/`threadIdx.x`：SIMT 内置线程索引变量，类似 GPU CUDA。全局线程 ID = `blockIdx.x * blockDim.x + threadIdx.x`。
- 越界检查 `if (idx >= index_total_length)` 是 SIMT 的必备模式——线程数通常向上取整为 Block 大小的整数倍，多出的线程需提前返回。
- `output[idx] = input[index[idx]]`：直接通过指针离散寻址读写，无需搬入 UB。这是 SIMT 与 SIMD 的本质区别——SIMD 通过基地址+步长批量寻址，无法处理随机地址；SIMT 每个线程独立寻址，天然适配 Gather/Scatter。


In [ ]:
!mkdir -p Sources/02.07

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("Environment ready.")


In [ ]:
%%writefile Sources/02.07/gather_1d.asc

#include <vector>
#include <iostream>
#include "acl/acl.h"

__global__ void gather_1d_custom(float* input, int32_t* index, float* output, uint64_t index_total_length)
{
    int32_t idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx >= index_total_length) {
        return;
    }
    output[idx] = input[index[idx]];
}



## Host 端代码实现
Host 端通过 `<<<>>>` 语法糖调用 Device 端核函数：

```cpp
// Configure kernel launch parameters.
// index.size() is 48 * 256, so 48 blocks and 256 threads per block
// cover one output element per thread.
uint32_t blocks_per_grid = 48;
uint32_t threads_per_block = 256;
uint32_t dyn_ubuf_size = 0;

gather_1d_custom<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(
    input_device, index_device, output_device, index.size());
```

SIMT 启动参数 `<<<grid_dim, block_dim, dynamic_memory_size, stream>>>` 有 4 个参数，比 SIMD 的 `<<<blockDim, 0, stream>>>` 多一个 `block_dim`（每块线程数）。本例启动 48 个 Block、每 Block 256 个线程，共 48×256=12288 个线程并行，每个线程处理 1 个输出元素。


In [ ]:
%%writefile -a Sources/02.07/gather_1d.asc

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t input_len = 100000;
    constexpr uint32_t index_len = 48 * 256;
    std::vector<float> input(input_len);
    for (uint32_t i = 0; i < input_len; i++) input[i] = i * 1.2f;
    std::vector<int32_t> index(index_len);
    for (uint32_t i = 0; i < index_len; i++) index[i] = (int32_t)((i * 97 + 13) % input_len);

    size_t input_bytes = input_len * sizeof(float);
    size_t index_bytes = index_len * sizeof(int32_t);
    size_t output_bytes = index_len * sizeof(float);

    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    float *input_device = nullptr;
    int32_t *index_device = nullptr;
    float *output_device = nullptr;
    uint8_t *output_host = nullptr;
    aclrtMallocHost((void**)&output_host, output_bytes);
    aclrtMalloc((void**)&input_device, input_bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&index_device, index_bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&output_device, output_bytes, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(input_device, input_bytes, input.data(), input_bytes, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(index_device, index_bytes, index.data(), index_bytes, ACL_MEMCPY_HOST_TO_DEVICE);

    uint32_t blocks_per_grid = 48;
    uint32_t threads_per_block = 256;
    uint32_t dyn_ubuf_size = 0;

    gather_1d_custom<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(
        input_device, index_device, output_device, index_len);

    aclrtSynchronizeStream(stream);
    aclrtMemcpy(output_host, output_bytes, output_device, output_bytes, ACL_MEMCPY_DEVICE_TO_HOST);

    std::vector<float> output((float*)output_host, (float*)(output_host + output_bytes));
    bool ok = true;
    for (uint32_t i = 0; i < index_len; i++) {
        if (output[i] != input[index[i]]) { ok = false; break; }
    }
    std::cout << (ok ? "[Success] verification passed." : "[Failed] verification failed!") << std::endl;

    aclrtFree(input_device);
    aclrtFree(index_device);
    aclrtFree(output_device);
    aclrtFreeHost(output_host);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}


## 编译与运行
SIMT 编译需加 `--enable-simt` 选项，本课程面向 950 使用 `--npu-arch=dav-3510`：


In [ ]:
!bisheng Sources/02.07/gather_1d.asc --npu-arch=dav-3510 --enable-simt -o Sources/02.07/demo


In [ ]:
!if [ -e /dev/davinci0 ]; then Sources/02.07/demo; else cannsim record Sources/02.07/demo -s Ascend950 -o Sources/02.07/sim_out; fi


## 课后实践
仿照本例用 SIMT 实现一维 Scatter：`output[index[i]] = input[i]`（注意离散写可能需要原子操作）。


In [ ]:
!cat ./answer/02.07_answer.txt
